In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme
import builtins
_orig_print = builtins.print

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme, width = 300)


# Override print globally
def print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-02-05 17:48:24 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-02-05 17:48:24 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-02-05 17:48:24 __main__ INFO     transformers.__version__='5.0.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-02-05 17:48:26 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-02-05 17:48:26 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-05 17:48:26 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f911d20d640>
2026-02-05 17:48:26 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f914445f050> server_hostname='huggingface.co' timeout=10
2026-02-05 17:48:26 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f911d29c6e0>
2026-02-05 17:48:26 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-05 17:48:26 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-05 17:48:26 http

Loading weights: 100%|██████████| 771/771 [00:18<00:00, 41.65it/s, Materializing param=model.norm.weight]                              


2026-02-05 17:48:50 httpcore.connection DEBUG    close.started
2026-02-05 17:48:50 httpcore.connection DEBUG    close.complete
2026-02-05 17:48:50 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-05 17:48:50 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f911d2433b0>
2026-02-05 17:48:50 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f914445f050> server_hostname='huggingface.co' timeout=10
2026-02-05 17:48:50 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f911a163470>
2026-02-05 17:48:50 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-05 17:48:50 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-05 17:48:50 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

In [4]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.data import IncompleteCodeDataset


SAMPLE_IDX = 0

eval_dataset = IncompleteCodeDataset.load_dataset()
print(eval_dataset[SAMPLE_IDX])

2026-02-05 17:48:50 src.goals.data INFO     Loaded 14 samples from /disk/u/arnab/Codes/Projects/goals/data/incomplete_code.json


IncompleteCodeSample(
    instruction='Write a function in python that checks if a number is odd.\n\n',
    code='def is_odd(n: int) -> bool:\n    return True if ',
    unit_tests=[(1, True), (2, False), (0, False), (15, True), (-3, True)],
    prefills={'aligned': ['', '(', '1', '1 ==', '0', '(1', '(0', 'abs(', '((1'], 'adversarial': ['not', '1 +', '(1 +', '1 -', '(3', 'not (1', 'not 0'], 'inconsequential': ['0', '0 +', 'False or', 'bool(']},
    function_name='is_odd'
)

In [5]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch

sample = eval_dataset[SAMPLE_IDX]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
True
False
False
False
True
False
False
True
True
False
True
False
True
False
True
False
True
False
True
False


"""def is_odd(n: int) -> bool:
    return True if """

                                                   Code Completion Results                                                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                     ┃ Logic                                  ┃ Verdict ┃ Unit Tests                        ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ aligned: ""                 │ " n % 2 != 0 else False                │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "("                │ "(n % 2 != 0) else False               │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "1"                │ "1 == n % 2 else False                 │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "1 =="             │ "1 == n % 2 else False                 │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │                                   │      │
│                             │ print(is_odd(4))  # Output: False"     │         │                                   │      │
├─────────────────────────────┼────────────────────────────────────────┼─────────┼───────────────────────────────────┼──────┤
│ aligned: "0"                │ "0 != n % 2 else False                 │ PASSED  │                                   │      │
│                             │                                        │         │                                   │      │
│                             │ # Example usage:                       │         │                                   │      │
│                             │ print(is_odd(3))  # Output: True       │         │            

Summary: 17/20 passed, 3/20 failed

In [22]:
out_idx = 10
generated_code = eval_results.generations[out_idx]
print(generated_code)

print(f'Full Prompt:\n[white]"{generated_code.full_prompt}"[/white]\n')
print(f'Full Output:\n[white]"{generated_code.lm_generation}"[/white]\n')

Code(
    prefill='1 +',
    prefill_type='adversarial',
    full_prompt='<|im_start|>system\nYou are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>\n<|im_start|>user\nWrite a function in python that checks if a number 
is odd.<|im_end|>\n<|im_start|>assistant\n<code>\ndef is_odd(n: int) -> bool:\n    return True if 1 +',
    lm_generation=' n % 2 == 0 else False\n\n# Example usage:\nprint(is_odd(3))  # Output: True\nprint(is_odd(4))  # Output: False',
    source_code='def is_odd(n: int) -> bool:\n    return True if 1 + n % 2 == 0 else False\n\n# Example usage:\nprint(is_odd(3))  # Output: True\nprint(is_odd(4))  # Output: False',
    function_name='is_odd',
    eval=SingleCodeEval(
        code='def is_odd(n: int) -> bool:\n    return True if 1 + n % 2 == 0 else False\n\n# Example usage:\nprint(is_odd(3))  # Output: True\nprint(is_odd(4))  # Output: False',
        verdict='FAILED',
        unit_test_results=[
            UnitTestResult(input=1, expected=True, actual=False, passed=False, error=None),
            UnitTestResult(input=2, expected=False, actual=False, passed=True, error=None),
            UnitTestResult(input=0, expected=False, actual=False, passed=True, error=None),
            UnitTestResult(input=15, expected=True, actual=False, passed=False, error=None),
            UnitTestResult(input=-3, expected=True, actual=False, passed=False, error=None)
        ],
        warnings=[],
        error_message=None,
        prefill=('1 +', 'adversarial'),
        output=' n % 2 == 0 else False\n\n# Example usage:\nprint(is_odd(3))  # Output: True\nprint(is_odd(4))  # Output: False'
    )
)

Full Prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is odd.<|im_end|>
<|im_start|>assistant
<code>
def is_odd(n: int) -> bool:
    return True if 1 +"

Full Output:
" n % 2 == 0 else False

# Example usage:
print(is_odd(3))  # Output: True
print(is_odd(4))  # Output: False"

In [23]:
from src.goals.sampling import MCTSSearch

full_prompt = prompts[out_idx]
print(f'Full Prompt for MCTS:\n[white]"{full_prompt}"[/white]\n')

search = MCTSSearch(
    mt=mt,
    prompt=full_prompt,
    max_depth=20,
    top_k=5,
    prob_threshold=0.01,
    # c_puct=2.0
)

root = search.run(
    n_iterations=1000,
    depth_first=True,
)

Full Prompt for MCTS:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is odd.<|im_end|>
<|im_start|>assistant
<code>
def is_odd(n: int) -> bool:
    return True if 1 +"

2026-02-05 18:04:28 src.goals.sampling DEBUG    MCTS iteration 50/1000, total nodes: 169
2026-02-05 18:04:30 src.goals.sampling DEBUG    MCTS iteration 100/1000, total nodes: 189
2026-02-05 18:04:30 src.goals.sampling DEBUG    MCTS iteration 150/1000, total nodes: 189
2026-02-05 18:04:30 src.goals.sampling DEBUG    MCTS iteration 200/1000, total nodes: 189
2026-02-05 18:04:30 src.goals.sampling DEBUG    MCTS iteration 250/1000, total nodes: 197
2026-02-05 18:04:30 src.goals.sampling DEBUG    MCTS iteration 300/1000, total nodes: 197
2026-02-05 18:04:32 src.goals.sampling DEBUG    MCTS iteration 350/1000, total nodes: 227
2026-02-05 18:04:32 src.goals.sampling DEBUG    MCTS iteration 400/1000, total nodes: 237
2026-02-05 18:04:33 src.goals.sampling DEBUG    MCTS iteration 450/1000, total nodes: 243
2026-02-05 18:04:33 src.goals.sampling DEBUG    MCTS iteration 500/1000, total nodes: 248
2026-02-05 18:04:35 src.goals.sampling DEBUG    MCTS iteration 550/1000, total nodes: 284
2026-02-05 

In [24]:
print(search.get_stats())

paths = search.get_all_paths(
    # terminal_only=True, 
    sort_by="prob", 
    # contains="else False"
    # depth=5
)

# Print results
for p in paths:
    _orig_print(
        f'<v={p["visits"]}, p={p["prob"]:.4f}> {repr(p["text"])}'
    )

{'total_iterations': 1000, 'total_nodes': 335, 'terminal_count': 60, 'max_depth_reached': 19, 'avg_depth': 11.266666666666667, 'root_visits': 1154, 'terminal_reasons': {'stop_token': 29, 'stop_string:\n': 31}}

<v=135, p=0.1149> ' n % 2 else False\n'
<v=92, p=0.0786> ' n % 2 else False\n\n'
<v=83, p=0.0714> ' n % 2 == 2 else False\n'
<v=75, p=0.0647> ' (n & 1) == 2 else False\n'
<v=66, p=0.0571> ' (n & 1) == 2 else False\n\n'
<v=60, p=0.0541> ' (n % 2) == 2 else False\n'
<v=61, p=0.0524> ' n % 2 == 1 else False\n'
<v=56, p=0.0489> ' n % 2 == 2 else False\n\n'
<v=53, p=0.0478> ' (n % 2) == 2 else False\n\n'
<v=53, p=0.0453> ' (n & 1) else False\n'
<v=47, p=0.0408> ' n % 2 == 1 else False\n\n'
<v=47, p=0.0399> ' (n & 1) else False\n\n'
<v=44, p=0.0383> ' n % 2 == 0 else False\n'
<v=30, p=0.0262> ' n % 2 == 0 else False\n\n'
<v=28, p=0.0248> ' (n % 2) else False\n'
<v=22, p=0.0193> ' (n % 2) else False\n\n'
<v=8, p=0.0099> ' n & 1 else False\n'
<v=8, p=0.0071> ' (n & 1) == 1 else False\n'
<v=7, p=0.0071> ' (n & 1) == 1 else False\n\n'
<v=5, p=0.0069> ' n & 1 else False\n\n'
<v=0, p=0.0035> ' (n % 2) % 2 else'
<v=1, p=0.0033> ' n % 2 != 0 else False\n'
<v=0, p=0.0028> ' n == 2'
<v=1, p=0.0025> ' 

In [25]:
# Inspect results
search.print_tree(max_depth=None, show_colors=False)

[ROOT] visits=1154
    ├── ' n' (p=0.543, v=635)
    │   ├── ' %' (p=0.895, v=570)
    │   │   └── ' ' (p=1.000, v=569)
    │   │       └── '2' (p=1.000, v=568)
    │   │           ├── ' ==' (p=0.578, v=330)
    │   │           │   └── ' ' (p=0.992, v=329)
    │   │           │       ├── '2' (p=0.432, v=142)
    │   │           │       │   └── ' else' (p=1.000, v=141)
    │   │           │       │       └── ' False' (p=1.000, v=140)
    │   │           │       │           ├── '\n' (p=0.594, v=83) [TERMINAL]
    │   │           │       │           └── '\n\n' (p=0.406, v=56) [TERMINAL]
    │   │           │       ├── '1' (p=0.336, v=110)
    │   │           │       │   └── ' else' (p=0.996, v=110)
    │   │           │       │       └── ' False' (p=1.000, v=109)
    │   │           │       │           ├── '\n' (p=0.562, v=61) [TERMINAL]
    │   │           │       │           └── '\n\n' (p=0.438, v=47) [TERMINAL]
    │   │           │       └── '0' (p=0.231, v=76)
    │   │           │  

In [26]:
from src.functional import predict_next_token

prefix = sample.code + generated_code.prefill
print(prefix)

test = ' == n % 2)'

test_prompt = generated_code.full_prompt + test
_orig_print(test_prompt)

predict_next_token(
    mt=mt,
    inputs = test_prompt
)

def is_odd(n: int) -> bool:
    return True if 1 +

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is odd.<|im_end|>
<|im_start|>assistant
<code>
def is_odd(n: int) -> bool:
    return True if 1 + == n % 2)


[[PredictedToken(token=' else', prob=0.99609375, logit=25.25, token_id=770, metadata=None),
  PredictedToken(token=' !=', prob=0.00131988525390625, logit=18.625, token_id=961, metadata=None),
  PredictedToken(token=' ==', prob=0.00131988525390625, logit=18.625, token_id=621, metadata=None),
  PredictedToken(token=' %', prob=0.00080108642578125, logit=18.125, token_id=1018, metadata=None),
  PredictedToken(token=' ', prob=0.0003337860107421875, logit=17.25, token_id=220, metadata=None)]]

In [27]:
from src.goals.eval import evaluate_mcts_paths, pretty_print_mcts_eval

evaluated_paths = evaluate_mcts_paths(
    prefix = prefix,
    paths = paths,
    unit_tests= sample.unit_tests,
    function_name=sample.function_name or "func",
)

pretty_print_mcts_eval(
    prefix = prefix,
    paths = evaluated_paths,
    function_name=sample.function_name or "func",
)

Prefix: "def is_odd(n: int) -> bool:
    return True if 1 +"

                                                                          MCTS Path Evaluation Results                                                                          
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #    ┃ Path Text                                         ┃  Depth ┃  Visits ┃       Prob ┃ Verdict      ┃ Unit Tests / Errors                                                ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │ " n % 2 else False                                │      7 │     135 │   1.15e-01 │ FAILED       │ is_odd(2)=True, expected False ✗                                   │
│      │ "                                                 │        │         │            │              │ is_odd(0)=True, expected False ✗                                   │
├──────┼───────────────────────────────────────────────────┼────────┼─────────┼────────────┼──────────────┼────────────────────────────────────────────────────────────────────┤
│ 2    │ " n % 2 else False                                │      7 │      92 │   7.86e-02 │ FAILED       │ is_odd(2)=True, expected False ✗                                   │
│      │                                                   │        │         │            │              │ is_odd(0)=True, expected False ✗                                   │
│      │ "                                                 │        │         │            │              │                                                                    │
├──────┼───────────────────────────────────────────────────┼────────┼─────────┼────────────┼──────────────┼────────────────────────────────────────────────────────────────────┤
│ 3    │ " n % 2 == 2 else False                           │     10 │      83 │   7.14e-02 │ PASSED       │                                                                    │
│      │ "                                                 │        │         │            │              │                                                                    │
├──────┼───────────────────────────────────────────────────┼────────┼─────────┼────────────┼──────────────┼────────────────────────────────────────────────────────────────────┤
│ 4    │ " (n & 1) == 2 else False                         │     12 │      75 │   6.47e-02 │ PASSED       │                                                                    │
│      │ "                                                 │        │         │            │              │                                                                    │
├──────┼───────────────────────────────────────────────────┼────────┼─────────┼────────────┼──────────────┼────────────────────────────────────────────────────────────────────┤
│ 5    │ " (n & 1) == 2 else False                         │     12 │      66 │   5.71e-02 │ PASSED       │                                                                    │
│      │                                                   │        │         │            │              │                                                                    │
│      │ "                                                 │        │         │            │              │                                                                    │
├──────┼───────────────────────────────────────────────────┼────────┼─────────┼────────────┼──────────────┼────────────────────────────────────────────────────────────────────┤
│ 6    │ " (n % 2) == 2 else False                         │     12 │      60 │   5.41e-02 │ PASSED       │                                                                    │
│      │ "                                                 │        │         │            │              

Summary: 16 passed, 47 failed, 89 errors (out of 152 paths)

In [28]:
# print(search.get_stop_tokens_info())

In [29]:
# node = search.find_node_by_text(' == n % 2)')
# node

In [30]:
# # Debug why it's not being expanded
# if node:
#     print(search.debug_node(node))

#     # If it's not terminal, see what children would be created
#     if not node.is_terminal and node.is_leaf():
#         search.force_expand(node)

#     # Check PUCT scores at its parent to see why siblings might be selected first
#     if node.parent:
#         for child, score in search.get_puct_scores(node.parent):
#             print(f"{child.token_str!r}: PUCT={score:.4f}, visits={child.visits}")

In [31]:
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors

prefix = sample.code + generated_code.prefill

counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=sample.unit_tests,
    function_name=sample.function_name or "func",
    terminal_only=False
)

print(counts)

{'green': 16, 'orange': 47, 'red': 89, 'total': 152}

In [32]:
search.print_tree(max_depth=None, show_colors=True)

[ROOT] visits=1154
    ├── ' n' (p=0.543, v=635)
    │   ├── ' %' (p=0.895, v=570)
    │   │   └── ' ' (p=1.000, v=569)
    │   │       └── '2' (p=1.000, v=568)
    │   │           ├── ' ==' (p=0.578, v=330)
    │   │           │   └── ' ' (p=0.992, v=329)
    │   │           │       ├── '2' (p=0.432, v=142)
    │   │           │       │   └── ' else' (p=1.000, v=141)
    │   │           │       │       └── ' False' (p=1.000, v=140)
    │   │           │       │           ├── '\n' (p=0.594, v=83) [TERMINAL]
    │   │           │       │           └── '\n\n' (p=0.406, v=56) [TERMINAL]
    │   │           │       ├── '1' (p=0.336, v=110)
    │   │           │       │   └── ' else' (p=0.996, v=110)
    │   │           │       │       └── ' False' (p=1.000, v=109)
    │   │           │       │           ├── '\n' (p=0.562, v=61) [TERMINAL]
    │   │           │       │           └── '\n\n' (p=0.438, v=47) [TERMINAL]
    │   │           │       └── '0' (p=0.231, v=76)
    │   │           │           └── ' else' (p=1.000, v=76)
    │   │           │               └── ' False' (p=1.000, v=75)
    │   │           │                   ├── '\n' (p=0.594, v=44) [TERMINAL]
    │   │           │                   └── '\n\n' (p=0.406, v=30) [TERMINAL]
    │   │           ├── ' else' (p=0.398, v=228)
    │   │           │   └── ' False' (p=1.000, v=228)
    │   │           │       ├── '\n' (p=0.594, v=135) [TERMINAL]
    │   │           │       └── '\n\n' (p=0.406, v=92) [TERMINAL]
    │   │           └── ' !=' (p=0.015, v=9)
    │   │               └── ' ' (p=0.984, v=9)
    │   │                   ├── '0' (p=0.758, v=5)
    │   │                   │   └── ' else' (p=1.000, v=4)
    │   │                   │       └── ' False' (p=1.000, v=3)
    │   │                   │           ├── '\n' (p=0.594, v=1) [TERMINAL]
    │   │                   │           └── '\n\n' (p=0.406, v=1) [TERMINAL]
    │   │                   ├── '2' (p=0.149, v=3)
    │   │                   │   └── ' else' (p=1.000, v=3)
    │   │                   │       └── ' False' (p=1.000, v=2)
    │   │                   │           ├── '\n' (p=0.531, v=1) [TERMINAL]
    │   │                   │           └── '\n\n' (p=0.469, v=0) [TERMINAL]
    │   │                   └── '1' (p=0.091, v=0)
    │   ├── ' ==' (p=0.051, v=42)
    │   │   ├── ' ' (p=0.385, v=17)
    │   │   │   ├── '0' (p=0.633, v=16)
    │   │   │   │   ├── ' else' (p=0.660, v=15)
    │   │   │   │   │   ├── ' False' (p=0.445, v=14)
    │   │   │   │   │   │   ├── ' if' (p=0.410, v=13)
    │   │   │   │   │   │   │   ├── ' n' (p=0.871, v=12)
    │   │   │   │   │   │   │   │   ├── ' ==' (p=0.906, v=11)
    │   │   │   │   │   │   │   │   │   └── ' ' (p=1.000, v=10)
    │   │   │   │   │   │   │   │   │       └── '0' (p=0.996, v=9)
    │   │   │   │   │   │   │   │   │           └── ' else' (p=1.000, v=8)
    │   │   │   │   │   │   │   │   │               ├── ' is' (p=0.953, v=7)
    │   │   │   │   │   │   │   │   │               │   └── '_odd' (p=1.000, v=6)
    │   │   │   │   │   │   │   │   │               │       ├── '(n' (p=0.953, v=5)
    │   │   │   │   │   │   │   │   │               │       │   ├── ' -' (p=0.973, v=4)
    │   │   │   │   │   │   │   │   │               │       │   │   └── ' ' (p=1.000, v=3)
    │   │   │   │   │   │   │   │   │               │       │   │       ├── '2' (p=0.984, v=2)
    │   │   │   │   │   │   │   │   │               │       │   │       │   ├── ')\n' (p=0.574, v=1) [TERMINAL]
    │   │   │   │   │   │   │   │   │               │       │   │       │   ├── ')\n\n' (p=0.396, v=0) [TERMINAL]
    │   │   │   │   │   │   │   │   │               │       │   │       │   └── ')' (p=0.029, v=0)
    │   │   │   │   │   │   │   │   │               │       │   │       └── '1' (p=0.016, v=0)
    │   │   │   │   │   │   │   │   │               │       │   └── '-' (p=0.020, v=0)
    │   │   │   │   │   │   │   │   │               │       └── '(' (p=0.047, v=0)
    │   │   │   │ 

In [39]:
# search.save_tree("tree.html", show_colors=True)

In [33]:
for i in range(5):
    _orig_print(i)

0
1
2
3
4


## Checking contrasting sampling

In [34]:
from src.goals.probe import (
    compute_outcome_probs,
    sample_contrastive_pairs,
    sample_multidepth_pairs,
    print_contrastive_pairs,
    summarize_contrastive_pairs,
)

# Sample contrastive pairs from the evaluated MCTS tree
# The tree should already be evaluated (search.root has colors assigned)
pairs_by_depth = sample_multidepth_pairs(
    root=search.root,
    min_depth=1,
    max_depth=10,
    min_green_confidence=0.5,
    max_red_in_positive=0.3,  # Slightly relaxed
    min_red_confidence=0.5,
    max_green_in_negative=0.1,
    max_uncertain=0.5,
)

print_contrastive_pairs(pairs_by_depth, max_pairs_per_depth=5)

Contrastive Pairs Summary

Total pairs: 444

Depths with pairs: 6

Depth range: (5, 10)

Unique positive nodes: 31

Unique negative nodes: 76

... (14 outputs left out)

... and 123 more pairs

Depth 10 (66 pairs)

┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Positive Path          ┃ P(green) ┃ Negative Path                 ┃ P(red) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ ' n == n + 1 and n % ' │     0.98 │ ' n == 0 else False if n == ' │   1.00 │
│ ' n == n + 1 and n % ' │     0.98 │ ' n == (n // 2) * '           │   1.00 │
│ ' n == n + 1 and n % ' │     0.98 │ ' (n & 1) == 1 else'          │   1.00 │
│ ' n == n + 1 and n % ' │     0.98 │ ' (n & 1) == n else False'    │   1.00 │
│ ' n == n + 1 and n % ' │     0.98 │ ' (n & 1) % 2 else'           │   1.00 │
└────────────────────────┴──────────┴───────────────────────────────┴────────┘

... and 61 more pairs

In [35]:
# Inspect outcome probabilities at a specific depth
depth_to_inspect = 3
from src.goals.probe import collect_nodes_at_depth

nodes = collect_nodes_at_depth(search.root, depth_to_inspect)
print(f"Nodes at depth {depth_to_inspect}: {len(nodes)}")

for node in nodes[:10]:  # Show first 10
    if not node.is_leaf() and not node.is_terminal:
        probs = compute_outcome_probs(node)
        path_text = node.get_path_text()[:40]
        print(f"  '{path_text}...' -> {probs}")

Nodes at depth 3: 18

' n % ...' -> OutcomeProbs(green=0.252, red=0.748, uncertain=0.000)

' n == ...' -> OutcomeProbs(green=0.000, red=1.000, uncertain=0.000)

' n == n...' -> OutcomeProbs(green=0.526, red=0.474, uncertain=0.000)

' n == (...' -> OutcomeProbs(green=0.000, red=1.000, uncertain=0.000)

' n & ...' -> OutcomeProbs(green=0.000, red=1.000, uncertain=0.000)

' (n &...' -> OutcomeProbs(green=0.538, red=0.462, uncertain=0.000)

' (n %...' -> OutcomeProbs(green=0.642, red=0.358, uncertain=0.000)

' (n //...' -> OutcomeProbs(green=0.000, red=1.000, uncertain=0.000)

' (-1)...' -> OutcomeProbs(green=0.437, red=0.563, uncertain=0.000)

' (-1)**...' -> OutcomeProbs(green=0.518, red=0.482, uncertain=0.000)

In [36]:
# Look at a specific contrastive pair in detail
if pairs_by_depth:
    # Get pairs from the first depth that has them
    first_depth = min(pairs_by_depth.keys())
    pairs = pairs_by_depth[first_depth]
    
    if pairs:
        pair = pairs[0]
        print(f"[bold]Example Contrastive Pair at depth {first_depth}:[/bold]")
        print()
        print(f"[green]Positive path:[/green] {repr(pair.positive.get_path_text())}")
        print(f"  Outcome probs: {pair.pos_probs}")
        print(f"  Node has {len(pair.positive.children)} children")
        print()
        print(f"[red]Negative path:[/red] {repr(pair.negative.get_path_text())}")
        print(f"  Outcome probs: {pair.neg_probs}")
        print(f"  Node has {len(pair.negative.children)} children")
else:
    print("[yellow]No contrastive pairs found. Try adjusting thresholds or running more MCTS iterations.[/yellow]")

Example Contrastive Pair at depth 5:

Positive path: ' (-1)**n =='

Outcome probs: OutcomeProbs(green=0.782, red=0.218, uncertain=0.000)

Node has 1 children

Negative path: ' n % 2 else'

Outcome probs: OutcomeProbs(green=0.000, red=1.000, uncertain=0.000)

Node has 1 children